# My Gen-Z Chadbot: run the app
Before you start: Runtime -> Change runtime type -> **T4 GPU**.
Run the cells **in order**. You need a free **ngrok** auth token (ngrok.com -> Your Authtoken).
A Serper key (serper.dev) is optional, only for the web-search toggle.

## 1. Install

In [ ]:
!pip install -q -U fastapi uvicorn pyngrok transformers peft accelerate bitsandbytes requests

## 2. Log in to Hugging Face

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Load Llama-3 + your adapter

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
ADAPTER_ID = "mohitsharma9761/my-genz-chadbot"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_ID)
model.eval()
print("Model ready")

## 4. Chat helper + the chat page

In [ ]:
import threading

SYSTEM_PROMPT = "You are Chad, a confident, funny chatbot that speaks in Gen-Z slang. Keep answers short, helpful and full of vibe."
lock = threading.Lock()
EOT = tokenizer.convert_tokens_to_ids("<|eot_id|>")

def generate_reply(messages, max_new_tokens=200):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + messages
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with lock, torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=0.7, top_p=0.9, eos_token_id=[tokenizer.eos_token_id, EOT],
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

HTML_PAGE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Chadbot</title>
<style>
  * { box-sizing: border-box; }
  body { margin:0; font-family: system-ui, sans-serif; background:#0f1117; color:#e8e8ee; display:flex; flex-direction:column; height:100vh; }
  header { padding:14px 18px; background:#171a23; border-bottom:1px solid #262a36; display:flex; justify-content:space-between; align-items:center; }
  header h1 { margin:0; font-size:18px; }
  header label { font-size:13px; color:#aab; cursor:pointer; }
  #chat { flex:1; overflow-y:auto; padding:18px; display:flex; flex-direction:column; gap:10px; }
  .msg { max-width:80%; padding:10px 14px; border-radius:14px; line-height:1.4; white-space:pre-wrap; word-wrap:break-word; }
  .user { align-self:flex-end; background:#5b5bf0; }
  .bot { align-self:flex-start; background:#222736; }
  form { display:flex; gap:8px; padding:12px; background:#171a23; border-top:1px solid #262a36; }
  input[type=text] { flex:1; padding:12px; border-radius:10px; border:1px solid #2d3242; background:#0f1117; color:#e8e8ee; font-size:15px; }
  button { padding:12px 18px; border:0; border-radius:10px; background:#5b5bf0; color:white; font-size:15px; cursor:pointer; }
  button:disabled { opacity:.5; cursor:not-allowed; }
</style>
</head>
<body>
<header>
  <h1>Chadbot \U0001F525</h1>
  <label><input type="checkbox" id="useSearch"> Web search</label>
</header>
<div id="chat"><div class="msg bot">Yooo, what's good? Ask me anything. \u2728</div></div>
<form id="form">
  <input type="text" id="input" placeholder="Type a message..." autocomplete="off">
  <button id="send" type="submit">Send</button>
</form>
<script>
  const chat = document.getElementById("chat");
  const form = document.getElementById("form");
  const input = document.getElementById("input");
  const send = document.getElementById("send");
  const useSearch = document.getElementById("useSearch");
  let history = [];

  function addMsg(text, who) {
    const d = document.createElement("div");
    d.className = "msg " + who;
    d.textContent = text;
    chat.appendChild(d);
    chat.scrollTop = chat.scrollHeight;
    return d;
  }

  form.addEventListener("submit", async (e) => {
    e.preventDefault();
    const text = input.value.trim();
    if (!text) return;
    input.value = "";
    addMsg(text, "user");
    const thinking = addMsg("...", "bot");
    send.disabled = true;
    try {
      const res = await fetch(useSearch.checked ? "/search" : "/chat", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ message: text, history: history.slice(-6) })
      });
      const data = await res.json();
      thinking.textContent = data.response;
      history.push({ role: "user", content: text });
      history.push({ role: "assistant", content: data.response });
    } catch (err) {
      thinking.textContent = "Bruh, something broke: " + err;
    }
    send.disabled = false;
    input.focus();
  });
</script>
</body>
</html>"""
print("ready")

## 5. Start the server
It will ask for your ngrok token (and an optional Serper key; just press Enter to skip that one).
When it finishes, **click the printed link** and chat!

In [ ]:
import json, requests, uvicorn
from getpass import getpass
from typing import List, Optional
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from pyngrok import ngrok

try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
except Exception:
    NGROK_AUTH_TOKEN = getpass("Paste your ngrok authtoken: ")

try:
    SERPER_API_KEY = userdata.get("SERPER_API_KEY")
except Exception:
    SERPER_API_KEY = getpass("Serper key (optional, press Enter to skip): ").strip() or None

app = FastAPI()

class ChatRequest(BaseModel):
    message: str
    history: Optional[List[dict]] = []

def clean_history(h):
    return [m for m in (h or []) if m.get("role") in ("user", "assistant") and isinstance(m.get("content"), str)][-6:]

@app.get("/", response_class=HTMLResponse)
def index():
    return HTML_PAGE

@app.post("/chat")
def chat(req: ChatRequest):
    messages = clean_history(req.history) + [{"role": "user", "content": req.message}]
    return {"response": generate_reply(messages, 200)}

@app.post("/search")
def search(req: ChatRequest):
    if not SERPER_API_KEY:
        return {"response": "No search key set, so web search is off. Turn the toggle off and just chat with me. \U0001F64F"}
    try:
        r = requests.post(
            "https://google.serper.dev/search",
            headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
            data=json.dumps({"q": req.message}), timeout=15,
        )
        context = "\n".join(x.get("snippet", "") for x in r.json().get("organic", [])[:5]) or "Nothing found."
    except Exception:
        context = "Web search is down bad rn."
    prompt = f"Use these web results: --- {context[:2000]} --- to answer: '{req.message}'. Keep it short and confident."
    return {"response": generate_reply([{"role": "user", "content": prompt}], 250)}

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000).public_url

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True,
).start()

print("\nYOUR CHADBOT IS LIVE:", public_url)
print("(ngrok may show a 'Visit Site' button the first time, just click it)")